# Build native Lden targets per city

Extracts each city's **published Lden** (day-evening-night composite) per road segment and writes
`results/<xxx>_lden.csv` (`road_id, lden_db`). **No recompute from binned day/evening/night** — the real
source field is used (recomputing from 5-dB bins is lossy).

| City | Lden source | method |
|---|---|---|
| Barcelona | `TOTAL_DEN` (`BCN_noise_streets.gpkg`, range strings) | re-join, `extract_min` |
| Viladecans | `TOTDEN` (enriched geojson, numeric) | midpoint nearest-join |
| Berlin | `GES_DEN_mean` = the regre dataset's `noise_day` | direct |
| Lyon | Lden raster = the regre dataset's `noise_day` (`db_day`) | direct |
| Zaragoza | `mapa_ruido_DEN_2016` WFS isophones (`DB_HI − 5`) | midpoint point-in-polygon |
| Milan | **no Lden** → diurnal zoning limit (`noise_day`) used as a **flagged proxy** | direct |

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import os, urllib.request
import warnings; warnings.filterwarnings('ignore')

os.makedirs('../results', exist_ok=True)

def extract_min(s):
    s = str(s)
    if s.startswith('< 40'):
        return 35
    return int(s.split(' - ')[0])

def regre_path(city, xxx):
    return f'../../{city}/notebooks/data/{xxx}_noise_regre_ml_dataset.csv'

lden = {}   # city -> DataFrame(road_id, lden_db)

## Berlin, Lyon, Milan — Lden already in the dataset
Berlin's `noise_day` is `GES_DEN_mean` (agency Lden); Lyon's is the Lden raster; Milan has no Lden so its
diurnal zoning limit stands in (flagged).

In [2]:
for city, xxx, note in [('Berlin','ber','GES_DEN (Lden)'),
                        ('Lyon','lyo','Lden raster'),
                        ('Milan','mil','zoning-limit PROXY (no real Lden)')]:
    d = pd.read_csv(regre_path(city, xxx))
    out = d[['road_id','noise_day']].rename(columns={'noise_day':'lden_db'}).copy()
    out['road_id'] = out['road_id'].astype(str)
    lden[city] = out
    out.to_csv(f'../results/{xxx}_lden.csv', index=False)
    print(f'{city:11s} {len(out):6d} segs | mean Lden {out.lden_db.mean():.1f} dB | {note}')

Berlin       70890 segs | mean Lden 52.2 dB | GES_DEN (Lden)
Lyon          7159 segs | mean Lden 64.9 dB | Lden raster
Milan        22164 segs | mean Lden 63.1 dB | zoning-limit PROXY (no real Lden)


## Barcelona — re-join `TOTAL_DEN`

In [3]:
edges = gpd.read_file('../../../notebooks/_elena/bcn_osmnx_edges.gpkg')
edges = edges[edges.geometry.type == 'LineString'].copy()
noise = gpd.read_file('../../../layers/BCN_noise_streets.gpkg').to_crs(edges.crs)

j = gpd.sjoin_nearest(edges[['segment_id','geometry']], noise[['TOTAL_DEN','geometry']],
                      max_distance=10, distance_col='dist')
j = j.dropna(subset=['TOTAL_DEN']).sort_values('dist').drop_duplicates('segment_id', keep='first')
j['lden_db'] = j['TOTAL_DEN'].apply(extract_min)

bcn = j[['segment_id','lden_db']].rename(columns={'segment_id':'road_id'}).copy()
bcn['road_id'] = bcn['road_id'].astype(str)
lden['Barcelona'] = bcn
bcn.to_csv('../results/bcn_lden.csv', index=False)
print('Barcelona  ', len(bcn), 'segs | mean Lden', round(bcn.lden_db.mean(),1), 'dB')

Barcelona   16212 segs | mean Lden 61.4 dB


## Viladecans — re-join `TOTDEN`

In [4]:
vs = gpd.read_file('../../Viladecans/layers/VIL_noise_streets.gpkg')
vn = gpd.read_file('../../Viladecans/layers/viladecans_noise_enriched.geojson').to_crs(vs.crs)

mid = gpd.GeoDataFrame(vs[['segment_id']].copy(),
                       geometry=vs.geometry.interpolate(0.5, normalized=True), crs=vs.crs)
j = gpd.sjoin_nearest(mid, vn[['TOTDEN','geometry']], distance_col='dist')
j = j[~j.index.duplicated(keep='first')]

vil = j[['segment_id','TOTDEN']].rename(columns={'segment_id':'road_id','TOTDEN':'lden_db'}).copy()
vil['road_id'] = vil['road_id'].astype(str)
vil['lden_db'] = vil['lden_db'].astype(float)
lden['Viladecans'] = vil
vil.to_csv('../results/vil_lden.csv', index=False)
print('Viladecans ', len(vil), 'segs | mean Lden', round(vil.lden_db.mean(),1), 'dB')

Viladecans  1613 segs | mean Lden 60.5 dB


## Zaragoza — download `mapa_ruido_DEN_2016` + midpoint join (`DB_HI − 5`)

In [5]:
DEN_ZIP = '../layers/zgz_ruido_den_2016.zip'
if not os.path.exists(DEN_ZIP):
    os.makedirs('../layers', exist_ok=True)
    url = ('https://idezar-sig.zaragoza.es/servicios/geoserver/mapa_del_ruido_2016/wfs'
           '?service=WFS&version=1.1.0&request=GetFeature'
           '&typeName=mapa_del_ruido_2016:mapa_ruido_DEN_2016&outputFormat=SHAPE-ZIP')
    urllib.request.urlretrieve(url, DEN_ZIP)
    print('downloaded DEN layer')

zs = gpd.read_file('../../Zaragoza/layers/ZGZ_noise_streets.gpkg')
den = gpd.read_file(DEN_ZIP).to_crs(zs.crs)
print('DEN isophones:', len(den), '| DB_HI:', sorted(den['DB_HI'].unique()))

mid = gpd.GeoDataFrame(zs[['segment_id']].copy(),
                       geometry=zs.geometry.interpolate(0.5, normalized=True), crs=zs.crs)
j = gpd.sjoin(mid, den[['DB_HI','geometry']], how='left', predicate='within')
j = j[~j.index.duplicated(keep='first')].sort_index()

fill = float(den['DB_HI'].min()) - 10   # below the lowest mapped band (matches the dia/tarde/noche convention)
lden_db = (j['DB_HI'].astype(float) - 5).fillna(fill)
zgz = pd.DataFrame({'road_id': zs['segment_id'].astype(str).values, 'lden_db': lden_db.values})
n_fill = int((j['DB_HI'].isna()).sum())
lden['Zaragoza'] = zgz
zgz.to_csv('../results/zgz_lden.csv', index=False)
print('Zaragoza   ', len(zgz), 'segs | mean Lden', round(zgz.lden_db.mean(),1),
      f'dB | {n_fill} below lowest band -> {fill:.0f}')

DEN isophones: 19764 | DB_HI: [np.float64(50.0), np.float64(55.0), np.float64(60.0), np.float64(65.0), np.float64(70.0), np.float64(75.0), np.float64(80.0), np.float64(85.0), np.float64(90.0)]
Zaragoza    14600 segs | mean Lden 64.5 dB | 0 below lowest band -> 40


## Verify — Lden vs day, and Barcelona spot-check

In [6]:
# Lden should sit at/above Lday for the genuine-3-period cities (night penalty), = day for Berlin/Lyon/Milan
for city, xxx in [('Barcelona','bcn'),('Viladecans','vil'),('Milan','mil'),
                  ('Berlin','ber'),('Lyon','lyo'),('Zaragoza','zgz')]:
    base = '../../../notebooks/_elena/data/bcn' if city=='Barcelona' else f'../../{city}/notebooks/data/{xxx}'
    d = pd.read_csv(base + '_noise_regre_ml_dataset.csv')[['road_id','noise_day']]
    d['road_id'] = d['road_id'].astype(str)
    m = d.merge(lden[city], on='road_id', how='left')
    cov = m['lden_db'].notna().mean()
    print(f'{city:11s} coverage {cov:5.1%} | mean day {m.noise_day.mean():.1f} -> Lden {m.lden_db.mean():.1f} dB '
          f'(delta {m.lden_db.mean()-m.noise_day.mean():+.1f})')

Barcelona   coverage 100.0% | mean day 59.7 -> Lden 61.2 dB (delta +1.6)
Viladecans  coverage 100.0% | mean day 59.9 -> Lden 60.5 dB (delta +0.6)
Milan       coverage 100.0% | mean day 63.1 -> Lden 63.1 dB (delta +0.0)


Berlin      coverage 100.0% | mean day 52.2 -> Lden 52.2 dB (delta +0.0)
Lyon        coverage 100.0% | mean day 64.9 -> Lden 64.9 dB (delta +0.0)
Zaragoza    coverage 100.0% | mean day 64.1 -> Lden 64.5 dB (delta +0.4)


In [7]:
# Barcelona spot-check: extracted TOTAL_DEN vs the source rows shown earlier (day 60-65 -> DEN 65-70 etc.)
print(lden['Barcelona'].head())
print('\nLden distribution (Barcelona):')
print(lden['Barcelona']['lden_db'].value_counts().sort_index())

                  road_id  lden_db
44  26057131_1362899977_0       70
45  26057304_3218700096_0       70
49    26057341_26057131_0       65
50    26057341_30242757_0       70
26    21638920_21638918_0       70

Lden distribution (Barcelona):
lden_db
35     126
40     101
45     415
50    1014
55    2656
60    3878
65    4772
70    3008
75     242
Name: count, dtype: int64
